In [ ]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

# Standard library imports
import argparse
import importlib.util
import inspect
import json
import math
import os
import pickle
import random
import shutil
import socket
import subprocess
import sys
import tempfile
import warnings
from functools import reduce

# Third-party imports
import librosa
import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from matplotlib import cm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from tqdm import tqdm
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
import umap
from pathlib import Path

from pytorch_grad_cam import GradCAM, HiResCAM, ScoreCAM, GradCAMPlusPlus, AblationCAM, XGradCAM, EigenCAM, FullGrad
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget, BinaryClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Local imports
import commons
import lightning_wrapper
import losses
import models
import utils
import train
from cough_datasets import (
    CoughDatasets,
    CoughDatasetsCollate,
    CoughDatasetsProcessorCollate,
    CoughDetectionRatioBatchSampler,
    CoughDiseaseBinaryBatchSampler,
    PatientBatchSampler
)

torch.set_float32_matmul_precision("medium")
cmap = cm.get_cmap("viridis")

def find_bilstmatt_logmel_logs(root="."):
    root = Path(root)
    log_folders = set()

    for p in root.rglob("bilstmatt_logmel"):
        if p.is_dir():
            parent = p.parent
            if parent.name == "logs" or parent.name.startswith("logs_"):
                log_folders.add(parent)

    return sorted(str(p) for p in log_folders)

class CAMWrapper(torch.nn.Module):
    def __init__(self, lightning_model):
        super().__init__()
        self.lightning_model = lightning_model

    def forward(self, x):
        out = self.lightning_model(x)
        return out["disease_logits"]

class SplitStreamCAMWrapper(nn.Module):
    def __init__(self, model, stream_idx: int):
        super().__init__()
        self.model = model
        self.stream_idx = stream_idx  # 0=raw, 1=delta, 2=deltadelta

    def forward(self, s):
        z = self.model.forward_encoder(s.unsqueeze(1))
        logits = self.model.classifier(z)
        return logits

def minmax_norm(x, eps=1e-8):
    return (x - x.min()) / (x.max() - x.min() + eps)

def reduce_embeddings(embeddings, method="umap", random_state=42):
    if method == "umap":
        reducer = umap.UMAP(
            n_neighbors=15,
            min_dist=0.1,
            n_components=2,
            random_state=random_state
        )
    elif method == "tsne":
        reducer = TSNE(
            n_components=2,
            perplexity=30,
            learning_rate="auto",
            init="pca",
            random_state=random_state
        )
    else:
        raise ValueError("method must be 'umap' or 'tsne'")
    
    return reducer.fit_transform(embeddings)


def scatter_plot(Z, color, title, cmap="viridis", cbar_label=None):
    plt.figure(figsize=(6, 5))
    sc = plt.scatter(Z[:, 0], Z[:, 1], c=color, cmap=cmap, s=12, alpha=0.8)
    plt.title(title)
    plt.xticks([])
    plt.yticks([])
    if cbar_label is not None:
        cbar = plt.colorbar(sc)
        cbar.set_label(cbar_label)
    plt.tight_layout()
    plt.show()

In [ ]:
experiments = ["logs_nfft2048/bilstmatt_logmel", "logs_nfft2048/bilstmatt_gtgram", "logs_nfft2048/bilstmatt_mfcc",
               "logs_nfft2048/resnet34re_logmel", "logs_nfft2048/resnet34re_gtgram", "logs_nfft2048/resnet34re_mfcc",
               "logs_ssl/qwenasp_peft", "logs_ssl/wavlmasp_peft"]
use_cpu = False

dfs = []
for now_experiment in experiments:
    experiment_metadata = now_experiment.split("/")
    parser = train.parse_args()
    args = parser.parse_args(["--model_name", experiment_metadata[1]])
    #log_folder = find_bilstmatt_logmel_logs(".")[0]
    model_dir = os.path.join(experiment_metadata[0], args.model_name)

    config_path = args.config_path if args.init else os.path.join(model_dir, "config.json")
    hps = train.load_config(config_path, model_dir, args)

    # =============================================================
    # SECTION: Loading Data
    # =============================================================
    df_train, df_test = train.load_data(hps)
    collate_fn = train.get_collate_fn(hps)
    target_labels = df_train[hps.data.target_column]

    logger = utils.get_logger(hps.model_dir, filename="dummy.log")
    logger.info(hps)

    pool_net, pool_model = train.setup_model(hps, is_init=args.init)
    runner_lightning = lightning_wrapper.CoughClassificationRunner.load_from_checkpoint(
        os.path.join(hps.model_dir, "best_model.ckpt"),
        model=pool_model,
        hps=hps, custom_logger=logger
    )
    runner_lightning.eval()
    trainer = L.Trainer(accelerator="gpu" if use_cpu == False else "cpu", devices="auto")

    with open(os.path.join(model_dir, "probs_threshold.pkl"), "rb") as f:
        runner_lightning.probs_threshold = pickle.load(f)['probs_threshold']

    info_fold_data = train.load_fold_info(hps.model_dir)
    best_fold_idx = info_fold_data.get("best_fold_idx", 0)

    train_dataset = CoughDatasets(
        df_train.values, 
        hps.data,
        wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{best_fold_idx}.pickle", 
        train=False
    )
    test_dataset = CoughDatasets(
        df_test.values, 
        hps.data,
        wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{best_fold_idx}.pickle", 
        train=False
    )
    
    # Create sampler
    #sampler = train.create_sampler(train_fold, hps)
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset, 
        num_workers=28, 
        #sampler=sampler, 
        batch_size=hps.train.batch_size,
        pin_memory=True, 
        collate_fn=collate_fn
    )
    test_loader = DataLoader(
        test_dataset, 
        num_workers=28, 
        shuffle=False, 
        batch_size=hps.train.batch_size,
        pin_memory=True, 
        collate_fn=collate_fn
    )

    test_wavnames = []
    test_probs = []
    test_preds = []
    test_labels = []
    #test_embeddings = []
    with torch.no_grad():
        for idx, batch in tqdm(enumerate(train_loader), total=len(train_loader)):
            wavnames, audio, _, attention_masks, dse_ids, [patient_ids, _, _] = batch
            audio = audio.cuda()
            attention_masks = attention_masks.cuda()
            out_model = runner_lightning.model.forward(audio, attention_mask=attention_masks)
            logits = out_model['disease_logits']

            probs = torch.sigmoid(logits).squeeze(-1)  # [B]
            preds = (probs >= runner_lightning.probs_threshold).long().cpu().detach().numpy()
            labels = torch.argmax(dse_ids, dim=1).cpu().detach().numpy()

            test_wavnames.extend(wavnames)
            test_labels.extend(labels)
            test_preds.extend(preds)
            test_probs.extend(probs.cpu().detach().numpy())
            #test_embeddings.extend(out_model['embeddings'].cpu().detach().numpy())

        for idx, batch in tqdm(enumerate(test_loader), total=len(test_loader)):
            wavnames, audio, _, attention_masks, dse_ids, [patient_ids, _, _] = batch
            audio = audio.cuda()
            attention_masks = attention_masks.cuda()
            out_model = runner_lightning.model.forward(audio, attention_mask=attention_masks)
            logits = out_model['disease_logits']

            probs = torch.sigmoid(logits).squeeze(-1)  # [B]
            preds = (probs >= runner_lightning.probs_threshold).long().cpu().detach().numpy()
            labels = torch.argmax(dse_ids, dim=1).cpu().detach().numpy()

            test_wavnames.extend(wavnames)
            test_labels.extend(labels)
            test_preds.extend(preds)
            test_probs.extend(probs.cpu().detach().numpy())
            #test_embeddings.extend(out_model['embeddings'].cpu().detach().numpy())

    del audio, attention_masks
    test_wavnames = np.array(test_wavnames)
    test_labels = np.array(test_labels)
    test_preds = np.array(test_preds)
    test_probs = np.array(test_probs)
    #test_embeddings = np.array(test_embeddings)

    df_now = pd.DataFrame({
        "path_file": test_wavnames,
        experiment_metadata[1]: test_preds
    })
    dfs.append(df_now)

final_df = reduce(
    lambda left, right: pd.merge(left, right, on="path_file", how="inner"),
    dfs
)

df_train['split'] = "train"
df_test['split'] = "test"

df_all = pd.concat([df_train, df_test])
df_all.reset_index(inplace=True, drop=True)

df_result = df_all.merge(
    final_df,
    on="path_file",
    how="inner",
    validate="one_to_one"
)

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score
)

def compute_metrics(y_true, y_pred, y_score=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

    metrics = {
        "Sensitivity": tp / (tp + fn) if (tp + fn) else 0.0,
        "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "Accuracy": accuracy_score(y_true, y_pred),
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
    }

    if y_score is not None and len(set(y_true)) > 1:
        metrics["ROC-AUC"] = roc_auc_score(y_true, y_score)
    else:
        metrics["ROC-AUC"] = float("nan")

    return metrics

def evaluate_file_level(df):
    y_true  = df["disease_status"].values
    y_pred  = df["bilstmatt_logmel"].values
    y_score = df["bilstmatt_logmel"].values  # binary but valid for AUC

    return compute_metrics(y_true, y_pred, y_score)


def evaluate_participant_voting(df):
    vote_df = (
        df.groupby("participant")
          .agg(
              disease_status=("disease_status", "first"),
              mean_score=("bilstmatt_logmel", "mean")
          )
    )

    y_true  = vote_df["disease_status"].values
    y_pred  = (vote_df["mean_score"] >= 0.5).astype(int)
    y_score = vote_df["mean_score"].values

    return compute_metrics(y_true, y_pred, y_score)

def compare_file_vs_participant(df):
    summary = pd.DataFrame.from_dict(
        {
            "File-level": evaluate_file_level(df),
            "Participant-voting": evaluate_participant_voting(df)
        },
        orient="index"
    )
    return summary

selected_df = df_result[df_result['split'] == "test"][['path_file', 'participant', 'disease_status', 'bilstmatt_logmel']]
summary_df = compare_file_vs_participant(selected_df)
print(summary_df)


In [ ]:
train_participants = set(df_train['participant'].unique())
test_participants = set(df_test['participant'].unique())

overlap_participants = train_participants.intersection(test_participants)

n_overlap = len(overlap_participants)
n_test_total = len(test_participants)

100 * n_overlap / n_test_total

# Check Error Overlap

In [ ]:
y_col = "disease_status"
participant_col = "participant"
split_col = "split"

symptom_cols = [
    "weight_loss", "hemoptysis", "night_sweats", "smoker"
]

meta_cols = ["gender"] + symptom_cols

model_cols = [
    "bilstmatt_logmel",
    "bilstmatt_gtgram",
    "bilstmatt_mfcc",
    "resnet34re_logmel",
    "resnet34re_gtgram",
    "resnet34re_mfcc",
    "qwenasp_peft",
    "wavlmasp_peft"
]

def compute_error_agreement(df):
    err = df[model_cols].ne(df[y_col], axis=0)
    df = df.copy()
    df["num_models_wrong"] = err.sum(axis=1)
    df["error_agreement_ratio"] = df["num_models_wrong"] / len(model_cols)
    return df

df_all = []
for split, df_s in df_result.groupby(split_col):
    df_s = compute_error_agreement(df_s)
    df_s["split_name"] = split
    df_all.append(df_s)

df_err = pd.concat(df_all, ignore_index=True)

In [ ]:
participant_summary = (
    df_err
    .groupby([split_col, participant_col])
    .agg(
        n_files=("path_file", "count"),
        mean_error_agreement=("error_agreement_ratio", "mean"),
        max_error_agreement=("error_agreement_ratio", "max")
    )
    .reset_index()
)

def participant_type(row):
    if row["mean_error_agreement"] > 0.6:
        return "systematically_ambiguous"
    elif row["max_error_agreement"] > 0.6:
        return "locally_ambiguous"
    else:
        return "consistently_easy"

participant_summary["participant_type"] = participant_summary.apply(
    participant_type, axis=1
)


In [ ]:
participant_summary

In [ ]:
file_vs_participant = (
    df_err
    .groupby([split_col, participant_col])
    .agg(
        files_high_agreement=("error_agreement_ratio", lambda x: (x > 0.6).sum()),
        total_files=("path_file", "count")
    )
)

file_vs_participant["fraction_files_ambiguous"] = (
    file_vs_participant["files_high_agreement"] /
    file_vs_participant["total_files"]
)

file_vs_participant.reset_index(inplace=True)

In [ ]:
file_vs_participant['fraction_files_ambiguous'].describe()

In [ ]:
participant_meta = (
    df_err
    .groupby([split_col, participant_col])
    .first()[meta_cols]
    .reset_index()
)

participant_analysis = participant_summary.merge(
    participant_meta,
    on=[split_col, participant_col],
    how="left"
)

gender_stats = (
    participant_analysis
    .groupby([split_col, "gender"])["mean_error_agreement"]
    .mean()
    .reset_index()
)

print(gender_stats)

for col in symptom_cols:
    stats = (
        participant_analysis
        .groupby([split_col, col])["mean_error_agreement"]
        .mean()
        .reset_index()
    )
    print(f"\nSymptom: {col}")
    print(stats)


In [ ]:
HIGH_AGREE = 0.6    # ≥60% of models wrong → systematic ambiguity
LOW_AGREE  = 0.0    # all models correct

summary_rows = []

for split, df_s in df_err.groupby(split_col):

    n_files = len(df_s)
    n_participants = df_s[participant_col].nunique()

    # File-level ambiguity
    frac_ambiguous_files = (df_s["error_agreement_ratio"] >= HIGH_AGREE).mean()
    frac_easy_files = (df_s["error_agreement_ratio"] == LOW_AGREE).mean()

    # Participant-level aggregation
    part = (
        df_s
        .groupby(participant_col)
        .agg(
            mean_err=("error_agreement_ratio", "mean"),
            frac_err=("error_agreement_ratio", lambda x: (x >= HIGH_AGREE).mean())
        )
    )

    # Whole particippant is likelywrong
    frac_systematic_participants = (part["mean_err"] >= HIGH_AGREE).mean()
    # Some of Files was Ambigu
    frac_participant_level = (part["frac_err"] >= 0.5).mean()

    # Gender correlation
    gender_gap = (
        df_s
        .groupby("gender")["error_agreement_ratio"]
        .mean()
    )
    gender_delta = (
        gender_gap.max() - gender_gap.min()
        if len(gender_gap) > 1 else np.nan
    )

    # Symptom correlation (mean absolute deltas)
    symptom_deltas = []
    for col in symptom_cols:
        g = df_s.groupby(col)["error_agreement_ratio"].mean()
        if len(g) > 1:
            symptom_deltas.append(g.max() - g.min())

    mean_symptom_delta = np.mean(symptom_deltas) if symptom_deltas else np.nan

    summary_rows.append({
        "split": split,
        "n_files": n_files,
        "n_participants": n_participants,

        # Core evidence
        "ambiguous_files_%": frac_ambiguous_files * 100,
        "easy_files_%": frac_easy_files * 100,

        "systematic_participants_%": frac_systematic_participants * 100,
        "participant_level_ambiguity_%": frac_participant_level * 100,

        # Metadata alignment
        "gender_effect_size": gender_delta,
        "symptom_effect_size": mean_symptom_delta
    })

summary_table = pd.DataFrame(summary_rows)
summary_table


In [ ]:
df_err

In [ ]:
df_e_t = df_err[df_err['split'] == "train"]
df_e_t[df_e_t["error_agreement_ratio"] >= 0.6]

# UMAP & TSNE

In [ ]:
for method in ["umap", "tsne"]:
    Z = reduce_embeddings(test_embeddings, method=method)
    scatter_plot(
        Z,
        color=test_labels,
        title=f"{method.upper()} – Colored by Ground Truth Labels",
        cmap="coolwarm",
        cbar_label="Label (0=Neg, 1=Pos)"
    )


In [ ]:
# Encode confusion categories
# 0: TN, 1: FP, 2: FN, 3: TP
conf_map = {
    (0, 0): 0,
    (0, 1): 1,
    (1, 0): 2,
    (1, 1): 3,
}

conf_labels = np.array([
    conf_map[(y, p)] for y, p in zip(test_labels, test_preds)
])

conf_names = ["TN", "FP", "FN", "TP"]
conf_cmap = plt.get_cmap("tab10")

for method in ["umap", "tsne"]:
    Z = reduce_embeddings(test_embeddings, method=method)
    scatter_plot(
        Z,
        color=conf_labels,
        title=f"{method.upper()} – Confusion Matrix View",
        cmap=conf_cmap,
        cbar_label="0:TN  1:FP  2:FN  3:TP"
    )


In [ ]:
threshold = runner_lightning.probs_threshold
test_probs = test_probs.squeeze()

# Signed distance to decision boundary
prob_margin = test_probs - threshold
for method in ["umap", "tsne"]:
    Z = reduce_embeddings(test_embeddings, method=method)
    scatter_plot(
        Z,
        color=prob_margin,
        title=f"{method.upper()} – Probability Margin (Threshold = {threshold:.2f})",
        cmap="coolwarm",
        cbar_label="p − threshold"
    )


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score

knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(test_embeddings, test_labels)
knn_preds = knn.predict(test_embeddings)

print("kNN BalAcc:", balanced_accuracy_score(test_labels, knn_preds))

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

k = 15
nbrs = NearestNeighbors(n_neighbors=k).fit(test_embeddings)
_, idx = nbrs.kneighbors(test_embeddings)

def label_entropy(labels):
    p = np.bincount(labels, minlength=2) / len(labels)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

local_entropy = np.array([
    label_entropy(test_labels[i_neighbors])
    for i_neighbors in idx
])

scatter_plot(
    Z,
    color=local_entropy,
    title="UMAP – Local Label Entropy",
    cmap="inferno",
    cbar_label="Label entropy"
)


In [ ]:
from scipy.stats import spearmanr

rho, p = spearmanr(test_probs, np.linalg.norm(test_embeddings, axis=1))
print(rho, p)

In [ ]:
# error indices for multiple models
err_sets = [set(err_idx_model_i), set(err_idx_model_j)]
jaccard = len(err_sets[0] & err_sets[1]) / len(err_sets[0] | err_sets[1])
print("Error overlap:", jaccard)


# XAI

In [ ]:
df_test_eval = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.test')
df_test_eval = df_test_eval.reset_index(drop=True)
df_test_eval = df_test_eval[hps.data.column_order]
#df_test_eval.drop(columns=["db"], inplace=True)

results_dict = train.evaluate_on_dataset(runner_lightning, trainer, df_test_eval, hps, best_fold_idx, collate_fn, db_column='db')

In [ ]:
preds = (test_df_thr["bilstmatt_logmel"] >= 0.5).astype(int).values
labels = test_df_thr["disease_status"].values
logits = test_df_thr["bilstmatt_logmel"].values

In [ ]:
cm = confusion_matrix(labels, preds, labels=[0, 1])
n_classes = cm.shape[0]

# sanity check: binary only
assert cm.shape == (2, 2), f"Expected binary confusion matrix, got {cm.shape}"

TN, FP = cm[0, 0], cm[0, 1]
FN, TP = cm[1, 0], cm[1, 1]

acc = accuracy_score(labels, preds)

# clinical metrics
sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0   # TB recall
spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0   # non-TB recall

In [ ]:
sens

In [ ]:
spec

In [ ]:
test_df_thr = df_result[df_result["split"] == "test"].copy()
test_df_thr["pred"] = (test_df_thr["bilstmatt_logmel"] >= 0.5).astype(int)

TP = ((test_df_thr.disease_status == 1) & (test_df_thr.pred == 1)).sum()
TN = ((test_df_thr.disease_status == 0) & (test_df_thr.pred == 0)).sum()
FP = ((test_df_thr.disease_status == 0) & (test_df_thr.pred == 1)).sum()
FN = ((test_df_thr.disease_status == 1) & (test_df_thr.pred == 0)).sum()

sensitivity = TP / (TP + FN + 1e-9)
specificity = TN / (TN + FP + 1e-9)

In [ ]:
sensitivity

In [ ]:
specificity

# Lets Find Out Why it Fail

In [ ]:
# Select Where bilstmatt_logmel always wrong and bilstmatt_mfcc always right
df_filtered = df_result[
    (df_result["split"] == "test") &
    (
        (df_result["bilstmatt_mfcc"] >= 0.5).astype(int)
        == df_result["disease_status"]
    ) &
    (
        (df_result["bilstmatt_logmel"] >= 0.5).astype(int)
        != df_result["disease_status"]
    )
]

In [ ]:
out_models = {}
for now_experiment in experiments:
    parser = train.parse_args()
    args = parser.parse_args(["--model_name", now_experiment])
    model_dir = os.path.join(log_folder, args.model_name)

    config_path = args.config_path if args.init else os.path.join(model_dir, "config.json")
    hps = train.load_config(config_path, model_dir, args)
    collate_fn = train.get_collate_fn(hps)

    logger = utils.get_logger(hps.model_dir, filename="dummy.log")
    logger.info(hps)

    pool_net, pool_model = train.setup_model(hps, is_init=args.init)
    runner_lightning = lightning_wrapper.CoughClassificationRunner.load_from_checkpoint(
        os.path.join(hps.model_dir, "best_model.ckpt"),
        model=pool_model,
        hps=hps, custom_logger=logger
    )
    runner_lightning.eval()
    info_fold_data = train.load_fold_info(hps.model_dir)
    best_fold_idx = info_fold_data.get("best_fold_idx", 0)

    test_dataset = CoughDatasets(
        df_filtered.values,
        hps.data,
        wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{best_fold_idx}.pickle",
        train=False
    )

    test_loader = DataLoader(
        test_dataset,
        num_workers=28,
        shuffle=False,
        batch_size=hps.train.batch_size,
        pin_memory=True,
        collate_fn=collate_fn
    )

    with torch.no_grad():
        for idx, batch in tqdm(enumerate(test_loader), total=len(test_loader)):
            wavnames, audio, _, attention_masks, dse_ids, [patient_ids, _, _] = batch
            audio = audio.cuda()
            attention_masks = attention_masks.cuda()
            out_model = runner_lightning.model.forward(audio, attention_mask=attention_masks)
            logits = out_model['disease_logits']

            probs = torch.sigmoid(logits).squeeze(-1)  # [B]
            #preds = (probs >= 0.5).long()
            dse_ids = torch.argmax(dse_ids, dim=1)
            break

        out_models[now_experiment] = {
            'audios': audio.cpu().detach(),
            'dse_ids': dse_ids.cpu().detach(),
            'out_model': out_model,
        }

In [ ]:
selected_index = random.randint(0, len(out_models['bilstmatt_logmel']['audios']) - 1)

selected_experiment = "bilstmatt_logmel"
for selected_experiment in out_models.keys():
    probs = torch.sigmoid(out_models[selected_experiment]['out_model']['disease_logits']).squeeze(-1)
    feature_model1 = minmax_norm(out_models[selected_experiment]['audios'][selected_index])
    t_att = out_models[selected_experiment]['out_model']['asp_weights'][selected_index].mean(dim=0).cpu().detach().numpy()
    se_att = out_models[selected_experiment]['out_model']['self_attn_weights'][selected_index].mean(dim=0).cpu().detach().numpy()
    time_x = np.arange(feature_model1.shape[-1])

    mel_np = np.flipud(feature_model1.numpy())
    cmap = cm.get_cmap("viridis")
    mel_rgb = cmap(mel_np)[..., :3]

    preds_prob = probs[selected_index]
    true_label = out_models[selected_experiment]['dse_ids'][selected_index]

    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 8), sharex=True)
    axes[0].imshow(mel_rgb, aspect='auto', extent=[time_x[0], time_x[-1], 0, mel_rgb.shape[0]])
    axes[0].set_title(f"{selected_experiment} - Label {true_label} - Pred Prob {preds_prob:.2f}")

    axes[1].plot(time_x, t_att)
    axes[1].set_title("Attentive Pooling Weight")
    axes[1].set_xlabel("Time Frame")

    axes[2].plot(time_x, se_att)
    axes[2].set_title("Mean Self Attention Weight")
    axes[2].set_xlabel("Time Frame")

    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 6), sharex=True)
time = np.arange(mel_norm.shape[-1])

axes[0].imshow(mel_rgb, aspect='auto', extent=[time[0], time[-1], 0, mel_rgb.shape[0]])
axes[0].set_title("Feature Raw")

axes[1].plot(time, t_att)
axes[1].set_title("Attention Weight")
axes[1].set_xlabel("Time Frame")

plt.tight_layout()
plt.show()

In [ ]:
selected_index = random.randint(0, len(audio) - 1)

In [ ]:
plt.plot(out_model['asp_weights'][selected_index].mean(dim=0).cpu().numpy())

In [ ]:
attn = out_model['self_attn_weights'][selected_index].detach().cpu().numpy()
plt.figure(figsize=(6, 5))
plt.imshow(attn, aspect="auto", origin="lower", cmap="viridis")
plt.colorbar(label="Attention weight")
plt.xlabel("Key time index")
plt.ylabel("Query time index")
plt.title("Temporal Self-Attention (Head-reduced)")
plt.tight_layout()
plt.show()

In [ ]:
plt.plot(out_model['self_attn_weights'][selected_index].mean(dim=0).detach().cpu().numpy())

In [ ]:
.shape

In [ ]:
selected_index

In [ ]:
out_model.keys()

In [ ]:
runner_lightning.probs_threshold = 0.6

In [ ]:
df_test_eval = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.test')
df_test_eval = df_test_eval.reset_index(drop=True)
df_test_eval = df_test_eval[hps.data.column_order + ['db']]

results_dict = evaluate_on_dataset(runner_lightning, trainer, df_test_eval, hps, best_fold_idx, collate_fn, db_column='db')
write_results_to_file(results_dict, model_dir, "Test Phase", db_map)

In [ ]:
torch.softmax(runner_lightning.model.fusion_logits, dim=0)

In [ ]:
val_batches = list(val_loader)

In [ ]:
wavnames, audio, _, attention_masks, dse_ids, [patient_ids, _, _] = random.choice(val_batches)
out_model = runner_lightning.model.forward(audio, attention_mask=attention_masks)
logits = out_model['disease_logits']

probs = torch.sigmoid(logits).squeeze(-1) 
preds = (probs >= 0.5).long().cpu().numpy()
labels = np.argmax(dse_ids.cpu().numpy(), axis=-1)

In [ ]:
selected_index = random.randint(0, len(audio) - 1)
mel = audio[selected_index].squeeze(0).cpu()

# split
mel_static = mel[0:80]
mel_delta = mel[80:160]
mel_deltadelta = mel[160:240]

# normalize independently
mel_static_n = minmax_norm(mel_static)
mel_delta_n = minmax_norm(mel_delta)
mel_deltadelta_n = minmax_norm(mel_deltadelta)

# recombine
mel_norm = np.vstack([
    mel_static_n,
    mel_delta_n,
    mel_deltadelta_n
])

mel_np = mel_norm
mel_np = np.flipud(mel_np)

cmap = cm.get_cmap("viridis")
mel_rgb = cmap(mel_np)[..., :3]

print(f"Pred : {preds[selected_index]}")
print(f"True : {labels[selected_index]}")
plt.imshow(mel_rgb)
plt.axis('off')
plt.tight_layout()
plt.show()

# GradCAM

In [ ]:
cam_model = CAMWrapper(runner_lightning.model)
target_layers = [runner_lightning.model.layer4, runner_lightning.model.layer3]
targets = [BinaryClassifierOutputTarget(labels[selected_index])]
input_tensor = audio[selected_index]#.squeeze(0)
image_original = mel_rgb

with GradCAM(model=cam_model, target_layers=target_layers) as cam:
  grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
  grayscale_cam = np.flipud(grayscale_cam[0, :])
  visualization = show_cam_on_image(image_original, grayscale_cam, use_rgb=True, image_weight=0.8)
  model_outputs = cam.outputs

images = [image_original, grayscale_cam, visualization]
titles = [f"GT",
            f"Heatmap (Pred",
            "Overlayed Heatmap"]

plt.figure(figsize=(21, 18))
for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.imshow(images[i])
    plt.axis('off')
    plt.title(titles[i], fontdict={"size": 18})
    plt.tight_layout()
plt.show()

In [ ]:
cam_model_split = SplitStreamCAMWrapper(runner_lightning.model, stream_idx=0)
target_layers = [runner_lightning.model.layer4, runner_lightning.model.layer3]

for i, name in enumerate(["raw", "delta", "deltadelta"]):
    streams = torch.split(input_tensor, 80, dim=1)
    s = streams[i]

    splits_original_image = np.split(image_original, 3, axis=0)
    with GradCAM(model=cam_model_split, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=s, targets=targets)
        grayscale_cam = np.flipud(grayscale_cam[0])
        visualization = show_cam_on_image(splits_original_image[i], grayscale_cam, use_rgb=True, image_weight=0.8)

    images = [splits_original_image[i], grayscale_cam, visualization]
    titles = [f"GT",
                f"Heatmap (Pred",
                "Overlayed Heatmap"]

    plt.figure(figsize=(21, 18))
    for i in range(3):
        plt.subplot(1, 3, i+1)
        plt.imshow(images[i])
        plt.axis('off')
        plt.title(titles[i], fontdict={"size": 18})
        plt.tight_layout()
    plt.show()


# Attention

In [ ]:
selected_index = random.randint(0, len(audio) - 1)
mel = audio[selected_index].squeeze(0).cpu()
t_att = out_model['time_attention'][selected_index].squeeze(-1).cpu().detach().numpy()

# split
mel_static = mel[0:80]
mel_static_n = minmax_norm(mel_static)

# recombine
mel_norm = np.vstack([
    mel_static_n,
])

mel_np = mel_norm
mel_np = np.flipud(mel_np)

cmap = cm.get_cmap("viridis")
mel_rgb = cmap(mel_np)[..., :3]

print(f"Pred : {preds[selected_index]}")
print(f"True : {labels[selected_index]}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 6))

axes[0].imshow(mel_rgb, aspect='auto')
axes[0].set_title("Feature Raw")

axes[1].plot(t_att)
axes[1].set_title("Attention Weight")

plt.tight_layout()
plt.show()